In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
import os
import math
import random
import numpy as np
import torch as ch
from glob import glob
from copy import deepcopy
from collections import Counter
from IPython.display import display

import sys
sys.path.append("../src/")
import utils
import matplotlib.pyplot as plt
from robustness_analyzer import RobustnessAnalyzer


from pytorch3d.io import load_obj 

### Camera Perturbation

In [ ]:
data_dir = "../data"

# Envmap(s) we're using during optimization (one or more)
max_envs = 5 # -1 or None for all
envmap_paths = glob(os.path.join("../data", "environments/*"))
random.shuffle(envmap_paths)
envmap_paths = envmap_paths[:max_envs]
print("#environments used:", len(envmap_paths))

true_class = 'container ship, containership, container vessel'
target_class = true_class
 
object_dir = "dredger"

params_to_optimize = ["camera"]

raster_settings = {
    "image_size": 224, # image resolution (image_size, image_size, 3)
    "bin_size": 16,  # Controls spatial partitioning for rasterization - larger values use less memory but may be slower
    "max_faces_per_bin": 100_000,  # Maximum faces per spatial bin - increase for complex meshes, decrease to save memory
}

kwargs = {
    "obj_path": os.path.join(data_dir, object_dir, "dredger.obj"),
    "texture_path": None,
    "envmap_paths": envmap_paths,
    "target_class": target_class,
    "batch_size": 4,  # How many different viewpoints we're optimizing in parallel (* #Environments)
    "params_to_optimize": params_to_optimize,
    "targeted": target_class!=true_class,
    "positive_z": True, # constraints camera z>0 (positive elevation)
    "raster_settings": raster_settings
}

robust_analyzer = RobustnessAnalyzer(**kwargs)

#### Run optimization: This will generate num_runs*batch_size viewpoints
num_runs = 50
num_iterations = 100
results_camera = robust_analyzer.run(num_runs, num_iterations, lr=5e-3)

In [ ]:
# Recover optimization data in case of an interrupted run
results_camera = robust_analyzer.get_current_results()

In [ ]:
labels_correct = ch.stack(results_camera["final_logits"]).reshape(-1, 1000).argmax(1) == utils.get_idx(true_class)

camera_positions = ch.stack([x["camera"] for x in results_camera["final_scene_params"]]).reshape(-1, 3)\
                        .repeat_interleave(len(envmap_paths) or 1, dim=0)
utils.visualize_positions_polar(camera_positions, labels_correct)

In [ ]:
utils.visualize_positions_with_distributions(camera_positions, labels_correct, mode="3d")

In [ ]:
utils.visualize_positions_with_distributions(camera_positions, labels_correct, mode="distributions")

If we want to re-render a specific output we can do it this way, here showing specific examples from the problematic areas from the distribution for Azimuth

In [ ]:
import torch

def display_rendered_images(robust_analyzer, results, run_index=0, env_index=0, image_indices=None,
                           max_cols=2, figsize=None, raster_settings={}):
    """
    Display rendered images with camera position information and prediction correctness.

    Args:
        model: The 3D model to render with
        results: Dictionary containing results from RobustnessAnalyzer
        run_index: Index of the run to display results from
        env_index: Index of the environment map to use
        image_indices: Indices of images to display (defaults to all)
        max_cols: Maximum number of columns in the grid
        figsize: Figure size (width, height) tuple
    """
    model = robust_analyzer.model

    # Ensure model has the updated parameters
    model.update_scene_params(results["final_scene_params"][run_index])

    # Render the images
    render_ims = model.render(with_grad=False, raster_settings=raster_settings)

    # Get predictions for the rendered images
    with torch.no_grad():
        logits = model()

    # Define image indices if not provided
    if image_indices is None:
        image_indices = range(robust_analyzer.batch_size)

    # Get target class index
    target_class_idx = utils.get_idx(robust_analyzer.target_class) if hasattr(robust_analyzer, 'target_class') else None

    # Calculate grid dimensions
    n_images = len(image_indices)
    n_rows = math.ceil(n_images / max_cols)

    # Create default figsize if not provided
    if figsize is None:
        figsize = (max_cols * 7, n_rows * 5)

    # Create the figure and axes
    fig, axes = plt.subplots(n_rows, max_cols, figsize=figsize)
    axes = np.atleast_1d(axes).flatten()

    # Display each image
    for ax, im_idx in zip(axes, image_indices):
        # Get camera position metrics
        camera_pos = results["final_scene_params"][run_index]["camera"][im_idx][None]
        azimuth, elevation, distance = utils.compute_spherical_coordinates(camera_pos)

        # Check if prediction is correct
        pred_idx = logits[im_idx, env_index].argmax().item()
        is_correct = (pred_idx == target_class_idx) if target_class_idx is not None else None
        pred_class = utils.get_target(pred_idx)

        # Create title with position and prediction info
        position_info = " ".join([
            f"{metric}={value.item():.1f}"
            for value, metric in zip([azimuth, elevation, distance], ["azimuth", "elevation", "distance"])
        ])

        if is_correct is not None:
            # Add color codes
            prediction_info = f"\n{pred_class[:20]}... ({'✓' if is_correct else '✗'})"
        else:
            prediction_info = f"\n{pred_class[:20]}..."

        title = position_info + prediction_info

        # Display image with title
        ax.imshow(utils.to_numpy(render_ims[im_idx][env_index]))
        ax.axis("off")
        ax.set_title(title)

    # Hide unused axes
    for ax in axes[len(image_indices):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
utils.display_rendered_images(
    robust_analyzer,
    results=results_camera,
    raster_settings={"image_size": 224},
    run_index=0,
    env_index=0 
)

In [ ]:
utils.create_logits_comparison_table(all_results={"camera": results_camera})